## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-08-17-51-53 +0000,nypost,Deal for 60 Wall St. bodes well for Lower Manh...,https://nypost.com/2026/11/08/business/deal-fo...
1,2026-11-08-17-46-40 +0000,nypost,Philly ‘Kristallnacht’ protests at Jewish-owne...,https://nypost.com/2026/11/08/us-news/philly-k...
2,2026-11-08-17-46-14 +0000,nyt,"Who Is Gadi Eisenkot, Israel’s Opposition Leader?",https://www.nytimes.com/2026/11/08/world/middl...
3,2026-11-08-17-40-37 +0000,nyt,Tesla Announces It Will Discontinue the Cybert...,https://www.nytimes.com/2026/11/08/us/tesla-cy...
4,2026-11-08-17-36-57 +0000,nypost,Bondi Sushi coming to 1221 Sixth Ave,https://nypost.com/2026/11/08/business/cuozzo-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2451/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
106,trump,31
153,white,16
229,new,14
105,president,13
37,democrats,12


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
159,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...,138
210,2026-11-07-18-59-01 +0000,nypost,Trump offers UFC boss Dana White role of White...,https://nypost.com/2026/11/07/us-news/trump-of...,114
161,2026-11-08-00-08-30 +0000,wapo,Trump says he offered White House press secret...,https://www.washingtonpost.com/politics/2026/1...,112
39,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...,105
15,2026-11-08-17-00-00 +0000,wsj,"Justice Department lawyer Stanley Woodward, a ...",https://www.wsj.com/us-news/law/how-a-former-d...,95


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
159,138,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...
57,66,2026-11-08-14-07-36 +0000,nypost,Netanyahu’s rival will have first shot at beco...,https://nypost.com/2026/11/08/us-news/israels-...
141,46,2026-11-08-02-00-00 +0000,wsj,A dominant midterm performance has given Democ...,https://www.wsj.com/politics/elections/the-new...
187,46,2026-11-07-21-01-09 +0000,nypost,Shocking footage shows moment Krakow police sh...,https://nypost.com/2026/11/07/world-news/shock...
214,43,2026-11-07-18-49-11 +0000,nypost,Marco Rubio warns top staff against conflicts ...,https://nypost.com/2026/11/07/us-news/marco-ru...
15,36,2026-11-08-17-00-00 +0000,wsj,"Justice Department lawyer Stanley Woodward, a ...",https://www.wsj.com/us-news/law/how-a-former-d...
201,36,2026-11-07-19-55-20 +0000,chitri,Israel’s president taps opposition candidate G...,https://www.chicagotribune.com/2026/11/07/eise...
39,35,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...
154,33,2026-11-08-00-43-48 +0000,nypost,Lunatic who killed 6 in Poland mass shooting h...,https://nypost.com/2026/11/07/world-news/very-...
136,28,2026-11-08-04-49-16 +0000,nypost,Ex-cop Sayed Mohassib had NYC jogger Katie Sua...,https://nypost.com/2026/11/07/us-news/ex-cop-s...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
